In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import math

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

class OutpaintTrainer:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        # Initialize Accelerator for FP16 mixed precision
        self.accelerator = Accelerator(mixed_precision='fp16')
        # Load model components
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()

        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj]
        self.vae, self.unet, self.coord_encoder, self.cond_proj = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr=5e-5)
        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")

        # Freeze VAE
        self.vae.requires_grad_(False)
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)

In [6]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

class OutpaintEngine:
    def __init__(self, device="cuda"):
        self.device = device

        # These must be loaded externally:
        self.vae = None
        self.unet = None
        self.accelerator = None
        self.noise_scheduler = None
        self.coord_encoder = None
        self.cond_proj = None

        self.directions = ['right', 'left', 'down', 'up']

        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])

    def _create_latent_mask(self, bbox, latent_shape):
        """
        Create latent mask for the diffusion process based on bbox.
        bbox shape: (B, 4) = (x1, y1, x2, y2) in normalized coords
        latent_shape: (B, C, H, W)
        """
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            x1 = coords[0] * lw
            y1 = coords[1] * lh
            x2 = coords[2] * lw
            y2 = coords[3] * lh

            xx, yy = torch.meshgrid(
                torch.arange(lw, device=self.device),
                torch.arange(lh, device=self.device)
            )
            mask = ((xx >= x1) & (xx <= x2) &
                    (yy >= y1) & (yy <= y2)).float()
            masks.append(mask)
        return torch.stack(masks).unsqueeze(1)

    def generate_iterative(self, image_tensor, steps=200, iterations=10, name="default"):
        """
        Iteratively fill a vertical gap in the center of an image
        by generating the central 64-pixel-wide region each time,
        shifting left/right in latent space, and filling the gap with white latent.
        """

        # Initial stitched image
        current_image = image_tensor.clone()
        b, c, h, w = current_image.shape

        # Define the central gap bbox in normalized coords
        bbox = torch.tensor([[0.0, 0.4375, 1.0, 0.5625]], device=self.device)

        # Initially encode to latent space
        with torch.no_grad():
            current_latent = self.vae.encode(current_image).latent_dist.sample()
            current_latent = current_latent * self.vae.config.scaling_factor

        for i in range(iterations):
            print(f"[{i+1}/{iterations}] Filling central gap in latent space")

            # Step 1: Create latent mask
            latent_mask = self._create_latent_mask(bbox, current_latent.shape)

            # Step 2: Masked latent
            masked_latent = current_latent * (1 - latent_mask)

            # Step 3: Add noise
            noise = torch.randn_like(current_latent)
            noisy_latent = self.noise_scheduler.add_noise(
                masked_latent * latent_mask,
                noise * latent_mask,
                torch.tensor(steps)
            )
            noisy_latent = masked_latent * (1 - latent_mask) + noisy_latent * latent_mask

            # Step 4: Denoising loop
            self.noise_scheduler.set_timesteps(steps)
            latent_input = noisy_latent

            condition = torch.cat([
                self.cond_proj(masked_latent),
                self.coord_encoder(bbox).unsqueeze(1).expand(-1, 64, -1)
            ], dim=-1)
            # 5) Denoising loop (keep context frozen outside gap)
            with torch.no_grad():
                for t in self.noise_scheduler.timesteps:
                    # enforce context outside mask
                    latent_input = latent_input * latent_mask + masked_latent * (1 - latent_mask)
                    noise_pred = self.unet(latent_input, t, encoder_hidden_states=condition).sample
                    latent_input = self.noise_scheduler.step(noise_pred, t, latent_input).prev_sample

                # --- IMPORTANT: compute the generated latent AFTER the loop ---
                generated_latent = latent_input * latent_mask + masked_latent * (1 - latent_mask)

            if i == iterations - 1:
              return generated_latent


            # Step 5: merge latent
            # extract the newly generated center region from latent_input
            new_patch_latent = latent_input[..., :, :, 28:36]   # latent pixels [28:36] = 64 image pixels

            # split into left and right halves (each 4 latent pixels)
            left_latent_patch = new_patch_latent[..., :, :, :4]   # (1, C, H, 4)
            right_latent_patch = new_patch_latent[..., :, :, 4:]  # (1, C, H, 4)

            # crop sides from current latent
            cropped_latent = current_latent[..., :, :, 4:-4]      # remove 4 latent pixels from each side → shape (1, C, H, 56)

            # split cropped latent into left and right parts
            cropped_left = cropped_latent[..., :, :, :24]         # left part
            cropped_right = cropped_latent[..., :, :, -24:]       # right part

            # create white gap latent (8 latent pixels = 64 image pixels)
            white_gap_latent = torch.zeros((1, current_latent.shape[1], current_latent.shape[2], 8), device=self.device)

            # concatenate parts:
            # cropped_left | left_patch | white_gap | right_patch | cropped_right
            stitched_latent = torch.cat([
                cropped_left,
                left_latent_patch,
                white_gap_latent,
                right_latent_patch,
                cropped_right
            ], dim=-1)

            # Crop back to 64 latent pixels width
            start = (stitched_latent.shape[-1] - 64) // 2
            stitched_latent = stitched_latent[..., :, :, start:start+64]

            # Update for next iteration
            current_latent = stitched_latent



In [5]:
trainer = OutpaintTrainer()
trainer.accelerator.load_state("drive/MyDrive/checkpoint_LR")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [8]:
import os
from PIL import Image
import numpy as np

outpaint_engine = OutpaintEngine()
outpaint_engine.unet = trainer.unet
outpaint_engine.cond_proj = trainer.cond_proj
outpaint_engine.coord_encoder = trainer.coord_encoder
outpaint_engine.vae = trainer.vae
outpaint_engine.accelerator = trainer.accelerator
outpaint_engine.noise_scheduler = trainer.noise_scheduler

# ======== CONFIG ========
image_name = "pair_0_8_leftright.png"   # <-- your one image
input_folder = input_folder = "drive/MyDrive/DISCO_USECASES/202509GapFillerOldResults/filltest"
image_path = os.path.join(input_folder, image_name)

save_folder = "/content/drive/MyDrive/0_8Bootstrap_latents/"
os.makedirs(save_folder, exist_ok=True)

N_RUNS = 20          # set 20..25 as you like
STEPS = 200
ITERATIONS = 6       # your current setting
BASE_SEED = 12345    # pick any; we’ll vary per run for reproducibility

# ======== LOAD & PREP IMAGE ONCE ========
img = Image.open(image_path).convert("RGB")
img_tensor = outpaint_engine.transform(img).unsqueeze(0).to(outpaint_engine.accelerator.device)

# ======== RUN MULTIPLE TIMES & SAVE ONLY LATENTS ========
for i in range(1, N_RUNS + 1):
    # set a unique, reproducible seed each run so results differ but are traceable
    seed = BASE_SEED + i
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    latent = outpaint_engine.generate_iterative(
        image_tensor=img_tensor,
        steps=STEPS,
        iterations=ITERATIONS,
        name=f"run{i}"
    )

    # Save just the latent; include metadata so you remember how it was made
    save_path = os.path.join(save_folder, f"{i}.pt")
    torch.save(
        {
            "latent": latent.cpu(),      # tensor (1, C, H, W)
            "seed": seed,
            "steps": STEPS,
            "iterations": ITERATIONS,
            "image_name": image_name,
        },
        save_path
    )

print(f"Saved {N_RUNS} latents to {save_folder}")



[1/6] Filling central gap in latent space
[2/6] Filling central gap in latent space
[3/6] Filling central gap in latent space
[4/6] Filling central gap in latent space
[5/6] Filling central gap in latent space
[6/6] Filling central gap in latent space
[1/6] Filling central gap in latent space
[2/6] Filling central gap in latent space
[3/6] Filling central gap in latent space
[4/6] Filling central gap in latent space
[5/6] Filling central gap in latent space
[6/6] Filling central gap in latent space
[1/6] Filling central gap in latent space
[2/6] Filling central gap in latent space
[3/6] Filling central gap in latent space
[4/6] Filling central gap in latent space
[5/6] Filling central gap in latent space
[6/6] Filling central gap in latent space
[1/6] Filling central gap in latent space
[2/6] Filling central gap in latent space
[3/6] Filling central gap in latent space
[4/6] Filling central gap in latent space
[5/6] Filling central gap in latent space
[6/6] Filling central gap in laten